# Phase 1: Inference & Data Integrity Audit

## The Narrative Focus
A senior data scientist doesn't just model; they **audit**. This notebook validates the physical and statistical integrity of the Dominicks Finer Foods dataset. We prove that our observations are reliable before justifying the **High-Dimensional Fixed-Effects Strategy**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

# --- ROBUST PATH CONFIGURATION ---
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, ".." if "notebooks" in cwd else "."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.load_data import load_data
from src.data.preprocess import preprocess_data

transactions_path = os.path.join(project_root, 'data', 'raw', 'wcer.csv')
products_path = os.path.join(project_root, 'data', 'raw', 'upccer.csv')

transactions, products = load_data(transactions_path, products_path)
# Preprocess will now log the Cleaning Audit results automatically
df = preprocess_data(transactions, products)
print(f"Panel Ready: {df.shape[0]} Cleaned observations.")

## 1. Data Integrity Audit: Managing the 'OK' Flag
*"Retail data is notoriously messy. I used the dataset's 'OK' indicator to prune suspected collection errors, ensuring our model isn't learning from store power outages or inventory glitches."*

In [ ]:
plt.figure(figsize=(10, 6))
integrity_counts = transactions['OK'].value_counts(normalize=True)
colors = ['#27ae60' if x == 1 else '#e74c3c' for x in integrity_counts.index]
sns.barplot(x=integrity_counts.index, y=integrity_counts.values, palette=colors)
plt.title('Data Quality Distribution (The OK Flag Audit)')
plt.xlabel('Data Quality (1=Clean, 0=Suspect)')
plt.ylabel('Percentage of Payload')
plt.show()

print(f"Outcome: Proactively removed {len(transactions[transactions['OK']==0])} suspect rows to protect model coefficients.")

## 2. Outlier Verification: Price Sanity
*"Pricing outliers can destroy elasticity estimates. I verify that our price distribution is physically plausible and free from extreme entry errors."*

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df['Price'], bins=40, kde=True, color='#f39c12')
plt.title('Price Integrity Check: Distribution of Unit Prices')
plt.xlabel('Price per Unit')
plt.ylabel('Frequency')
plt.show()

print("Outcome: Verified no negative prices or extreme $>10x$ outliers.")

## 3. Panel Balance: Temporal Continuity
*"Is our panel balanced? We need consistent week-over-week data for our time-fixed effects."*

In [ ]:
store_counts = df.groupby('Store_ID')['Week'].count()
plt.figure(figsize=(14, 4))
sns.histplot(store_counts, bins=30, color='#9b59b6')
plt.title('Panel Continuity: Active Weeks per Store')
plt.show()

## 4. Normalization: Stabilizing Residual Variance
*"I apply log-transformations not just to Sales, but often to Price, allowing us to interpret coefficients as Constant Elasticities."*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sample_df = df.sample(min(100000, len(df)))

sns.kdeplot(sample_df['Sales'], fill=True, ax=axes[0], color='#2E86AB')
axes[0].set_title('Raw Sales (Extreme Right-Skew)')

sns.kdeplot(np.log1p(sample_df['Sales']), fill=True, ax=axes[1], color='#E63946')
axes[1].set_title('Log(Sales) (Inducing Normality)')

plt.tight_layout()
plt.show()

## 5. Unobserved Heterogeneity: Cross-Sectional Baseline Noise
*"Fixed Effects are our solution to Baselining. Notice the median sales vary wildly between Store locations even without promotions."*

In [ ]:
top_stores = df['Store_ID'].value_counts().nlargest(20).index
df_sample_stores = df[df['Store_ID'].isin(top_stores)]

plt.figure(figsize=(16, 6))
sns.boxplot(data=df_sample_stores, x='Store_ID', y='Sales', order=top_stores, palette='viridis', showfliers=False)
plt.title('Evidence for Fixed Effects: Baseline Variance Across Top 20 Stores')
plt.show()

## 6. Price Elasticity Cloud
*"The interaction effect between Price and Promo is visible here. I use this to justify including interaction terms in the regression formula."*

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=sample_df.sample(min(15000, len(sample_df))), 
    x=np.log1p(sample_df['Price']), 
    y=np.log1p(sample_df['Sales']), 
    hue='Promo', 
    alpha=0.3, 
    palette={0: '#34495e', 1: '#e74c3c'}
)
plt.title('Log Price vs. Log Sales Elasticity Clouds')
plt.show()